## Put your name and student ID here:

**Name:** Laurie Gantuangco

**Student ID:** 1771419

## AI-assistance disclosure

GitHub Copilot (agent mode in VS Code) was used for this notebook. It helped in the following ways:

- **Implementing the TODOs.** It wrote the code for every TODO in the training loop: the GAN, cycle and identity loss criteria, the identity loss for both generators, the GAN and cycle losses for the horse→zebra and zebra→horse generators, and the real-image loss for each discriminator.
- **Explaining the code.** It wrote the `Solution` comments under each TODO that describe how that part works.
- **Running the notebook.** It ran the 20-epoch training cell on a local GPU and checked the translation images saved after each epoch. The provided settings were used unchanged.
- **Local setup.** The notebook was run locally instead of on Colab: the Google Drive mount cell was removed and `project_path` points to a local folder.

In [1]:
import os
import torch
import random
import itertools

import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt

from tqdm import tqdm
from PIL import Image
from pathlib import Path
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

## Defining the path to the project and dataset directories

In [2]:
project_path = Path('/home/laurie/CMPUT328/Assignment4')

dataset_path = project_path / 'datasets'

## Defining the Dataset for the unpaired images

The following unpaired image dataset implementation is provided to you. You do not need to change anything.

In [5]:
class UnpairedImageDataset(Dataset):
    def __init__(self, root, image_size=128):
        self.root = Path(root)

        self.horse_dir = self.root / 'trainA'
        self.zebra_dir = self.root / 'trainB'

        self.horse_paths = sorted(
            p for p in self.horse_dir.iterdir()
            if p.suffix.lower() in ['.jpg', '.jpeg', '.png']
        )


        self.zebra_paths = sorted(
            p for p in self.zebra_dir.iterdir()
            if p.suffix.lower() in ['.jpg', '.jpeg', '.png']
        )

        if not self.horse_paths or not self.zebra_paths:
            raise RuntimeError(
                'No images found. Check trainA and trainB directories.'
            )

        # Normalize RGB values to [-1, 1].
        self.transform = transforms.Compose([
            transforms.Resize(
                (image_size, image_size),
                interpolation=transforms.InterpolationMode.BICUBIC
            ),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.5, 0.5, 0.5),
                std=(0.5, 0.5, 0.5)
            )
        ])

        print(f'Horses: {len(self.horse_paths)}')
        print(f'Zebras: {len(self.zebra_paths)}')

    def __len__(self):
        return max(len(self.horse_paths), len(self.zebra_paths))

    def __getitem__(self, index):
        horse_path = self.horse_paths[
            index % len(self.horse_paths)
        ]

        # Sample independently: no paired correspondence is assumed.
        zebra_path = random.choice(self.zebra_paths)

        with Image.open(horse_path) as img:
            horse = self.transform(img.convert('RGB'))

        with Image.open(zebra_path) as img:
            zebra = self.transform(img.convert('RGB'))

        return {
            'horse': horse,
            'zebra': zebra
        }


class ReplayBuffer:
    def __init__(self, max_size=50):
        self.max_size = max_size
        self.images = []

    def push_and_pop(self, batch):
        returned = []

        for image in batch.detach():
            image = image.unsqueeze(0)

            if len(self.images) < self.max_size:
                self.images.append(image)
                returned.append(image)
            elif random.random() > 0.5:
                index = random.randrange(self.max_size)

                old_image = self.images[index].clone()
                self.images[index] = image
                returned.append(old_image)
            else:
                returned.append(image)

        return torch.cat(returned, dim=0)

def denormalize(images):
    return (images * 0.5 + 0.5).clamp(0, 1)


## Defining the CycleGan architecture

The full implementations of the CycleGAN's generators and discriminators are provided for you. No need to change anything.

In [6]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(
                channels, channels,
                kernel_size=3, stride=1, padding=0,
                bias=False
            ),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),

            nn.ReflectionPad2d(1),
            nn.Conv2d(
                channels, channels,
                kernel_size=3, stride=1, padding=0,
                bias=False
            ),
            nn.InstanceNorm2d(channels)
        )

    def forward(self, x):
        return x + self.block(x)

class Generator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3,
                 base_channels=64, num_residual_blocks=6):
        super().__init__()

        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(
                in_channels, base_channels,
                kernel_size=7, padding=0, bias=False
            ),
            nn.InstanceNorm2d(base_channels),
            nn.ReLU(inplace=True)
        ]

        # Downsampling: H x W -> H/4 x W/4
        channels = base_channels

        for _ in range(2):
            layers.extend([
                nn.Conv2d(
                    channels, channels * 2,
                    kernel_size=3, stride=2, padding=1,
                    bias=False
                ),
                nn.InstanceNorm2d(channels * 2),
                nn.ReLU(inplace=True)
            ])
            channels *= 2

        # Residual blocks
        for _ in range(num_residual_blocks):
            layers.append(ResidualBlock(channels))

        # Upsampling: H/4 x W/4 -> H x W
        for _ in range(2):
            layers.extend([
                nn.Upsample(scale_factor=2, mode="nearest"),
                nn.ReflectionPad2d(1),
                nn.Conv2d(
                    channels, channels // 2,
                    kernel_size=3, stride=1, padding=0,
                    bias=False
                ),
                nn.InstanceNorm2d(channels // 2),
                nn.ReLU(inplace=True)
            ])
            channels //= 2

        layers.extend([
            nn.ReflectionPad2d(3),
            nn.Conv2d(
                channels, out_channels,
                kernel_size=7, padding=0
            ),
            nn.Tanh()
        ])

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

class Discriminator(nn.Module):
    def __init__(self, in_channels=3, base_channels=64):
        super().__init__()

        layers = [
            nn.Conv2d(
                in_channels, base_channels,
                kernel_size=4, stride=2, padding=1
            ),
            nn.LeakyReLU(0.2, inplace=True)
        ]

        channels = base_channels

        for multiplier, stride in [(2, 2), (4, 2), (8, 1)]:
            next_channels = base_channels * multiplier

            layers.extend([
                nn.Conv2d(
                    channels, next_channels,
                    kernel_size=4, stride=stride, padding=1,
                    bias=False
                ),
                nn.InstanceNorm2d(next_channels),
                nn.LeakyReLU(0.2, inplace=True)
            ])

            channels = next_channels

        # No sigmoid: LSGAN uses raw scores.
        layers.append(
            nn.Conv2d(
                channels, 1,
                kernel_size=4, stride=1, padding=1
            )
        )

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

def init_weights(module):
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

        if module.bias is not None:
            nn.init.zeros_(module.bias)

def setup_cyclegan(device):
    G_H2Z = Generator().to(device)
    G_Z2H = Generator().to(device)

    D_H = Discriminator().to(device)
    D_Z = Discriminator().to(device)

    for model in [G_H2Z, G_Z2H, D_H, D_Z]:
        model.apply(init_weights)

    return G_H2Z, G_Z2H, D_H, D_Z


## Training - Defining visualization utility functions

The following functions are provided to you. You don't need to change anything.

In [7]:
def save_training_images(
    images,
    path,
    name
):
    fig, ax = plt.subplots(len(images) // 3, 3)

    titles = [
        ['Real Horse', 'Fake Zebra', 'Reconstructed Horse'],
        ['Real Zebra', 'Fake Horse', 'Reconstructed Zebra']
      ]

    for i in range(len(images)):
        row = i // 3
        col = i % 3
        ax[row, col].imshow(images[i].permute(1, 2, 0).detach().cpu().numpy())
        ax[row, col].set_title(titles[row][col])

    fig.savefig(os.path.join(path, name))
    plt.close()

def save_translations(G_H2Z, G_Z2H, sample_horse, sample_zebra, epoch):
    """Save translations and cycle reconstructions for fixed images."""

    G_H2Z.eval()
    G_Z2H.eval()

    with torch.no_grad():
        fake_zebra = G_H2Z(sample_horse)
        reconstructed_horse = G_Z2H(fake_zebra)

        fake_horse = G_Z2H(sample_zebra)
        reconstructed_zebra = G_H2Z(fake_horse)

        images = torch.cat([
            sample_horse,
            fake_zebra,
            reconstructed_horse,
            sample_zebra,
            fake_horse,
            reconstructed_zebra
        ], dim=0)

        images = denormalize(images)

        save_training_images(
            images,
            output_dir,
            f'epoch_{epoch:03d}.png'
        )

    G_H2Z.train()
    G_Z2H.train()

def save_checkpoint(G_H2Z, G_Z2H, D_H, D_Z, optimizer_G, optimizer_D_H, optimizer_D_Z, epoch):
    checkpoint = {
        'epoch': epoch,
        'G_H2Z': G_H2Z.state_dict(),
        'G_Z2H': G_Z2H.state_dict(),
        'D_H': D_H.state_dict(),
        'D_Z': D_Z.state_dict(),
        'optimizer_G': optimizer_G.state_dict(),
        'optimizer_D_H': optimizer_D_H.state_dict(),
        'optimizer_D_Z': optimizer_D_Z.state_dict()
    }

    torch.save(
        checkpoint,
        checkpoint_dir / f'checkpoint_{epoch:03d}.pt'
    )

## Training - Training Loop

### **Part 1: Completing the loss criteria (GAN, Cycle, Identity) (6%)**
The first thing you need to complete is the definitions of ```criterion_GAN, criterion_identity, criterion_cycle``` to be used to train the generators and discriminators.

### **Part 2: Applying the identity loss in training for both generators (20%)**
One feature of the CycleGAN's generators is that they should not translate an image already belonging to the target distribution; i.e., a Horse-t-Zebra generator should not do anything if it receives a zebra image.

### **Part 3: Applying the GAN and cycle losses for Horse->Zebra generator (25%)**
Horse->Zebra generator must produce images that can fool the zebra discriminator. Moreover, if we translate that image back to a horse using Zebra->Horse generator we should get the same horse image as we initially had.

### **Part 4: Applying the GAN and cycle losses for Zebra->Horse generator (25%)**
Zebra->Horse generator must produce images that can fool the horse discriminator. Moreover, if we translate that image back to a zebra using Horse->Zebra generator we should get the same zebra image as we initially had.

### **Part 5: Applying the discriminator loss for zebra discriminator (9%)**
Zebra discriminator must not be easilly fooled so it should be trained to distinguish the real zebra image from the fake one.

### **Part 6: Applying the discriminator loss for horse discriminator (9%)**
Horse discriminator must not be easilly fooled so it should be trained to distinguish the real horse image from the fake one.


NOTE: ReplayBuffer is an implementation trick used to make the training more stable and to make the discriminator not converge too quickly before the generators can learn properly. You don't need to change that part.

### **Part 7: Check the visual results (6%)**
You will get the mark for this part if your model can visually translate at least one animal to the other.

In [ ]:
epochs = 20
save_every = 1
device = 'cuda'
batch_size = 1
lr = 2e-4
lambda_cycle = 2
lambda_identity = 1

output_dir = project_path / 'outputs' / 'cyclegan'
output_dir.mkdir(parents=True, exist_ok=True)

checkpoint_dir = output_dir / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Define the generators and discriminators
G_H2Z, G_Z2H, D_H, D_Z = setup_cyclegan(device)

# Defining the dataset and dataloaders
dataset = UnpairedImageDataset(dataset_path / 'horse2zebra')
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# TODO: Define the loss functions
# Solution:
# - GAN loss: least-squares (LSGAN) objective, i.e. MSE between the
#   discriminator's raw patch scores and a target of 1 (real) or 0 (fake).
#   The Discriminator has no sigmoid, so it outputs unbounded scores, which
#   is what MSE expects (BCE would need probabilities). The same criterion
#   is reused by the provided fake-image discriminator terms below.
# - Cycle and identity losses: L1 between images. Both compare a generated
#   image to a real one pixel by pixel, and L1 gives sharper results than L2.
criterion_GAN = nn.MSELoss()
criterion_cycle = nn.L1Loss()
criterion_identity = nn.L1Loss()

# Defining the optimizer for both generators
optimizer_G = torch.optim.Adam(
    itertools.chain(
        G_H2Z.parameters(),
        G_Z2H.parameters()
    ),
    lr=lr,
    betas=(0.5, 0.999)
)

# Defining the optimizer for horse discriminator
optimizer_D_H = torch.optim.Adam(
    D_H.parameters(),
    lr=lr,
    betas=(0.5, 0.999)
)

# Defining the optimizer for zebra discriminator
optimizer_D_Z = torch.optim.Adam(
    D_Z.parameters(),
    lr=lr,
    betas=(0.5, 0.999)
)

# Defining the replay buffers
fake_horse_buffer = ReplayBuffer()
fake_zebra_buffer = ReplayBuffer()

# Keep track of the loss
history = {
    'G': [],
    'D_H': [],
    'D_Z': [],
    'cycle': [],
    'identity': []
}

# Main training loop
for epoch in range(1, epochs + 1):

    G_H2Z.train()
    G_Z2H.train()
    D_H.train()
    D_Z.train()

    running = {
        'G': 0.0,
        'D_H': 0.0,
        'D_Z': 0.0,
        'cycle': 0.0,
        'identity': 0.0
    }

    progress = tqdm(
        dataloader,
        desc=f'Epoch {epoch}/{epochs}'
    )

    for batch in progress:
        real_horse = batch['horse'].to(device)
        real_zebra = batch['zebra'].to(device)

        # 1. Update generators
        optimizer_G.zero_grad(set_to_none=True)

        # TODO: Training with the identity loss for both generators
        # Solution: feed each generator an image that is ALREADY in its output
        # domain. G_Z2H outputs horses, so it gets the real horse; G_H2Z
        # outputs zebras, so it gets the real zebra. The loss below then
        # penalizes any change, teaching the generators to leave such images
        # alone (which also preserves colours in normal translations).
        identity_horse = G_Z2H(real_horse)
        identity_zebra = G_H2Z(real_zebra)

        loss_identity = (
            criterion_identity(identity_horse, real_horse)
            + criterion_identity(identity_zebra, real_zebra)
        )

        # 2. Horse -> Zebra -> Horse
        # TODO: Training the zebra generator
        # Solution:
        # a) Translate the real horse into a fake zebra.
        fake_zebra = G_H2Z(real_horse)

        # b) Score it with the zebra discriminator. No detach here: the
        #    gradient must flow through D_Z back into G_H2Z.
        pred_fake_zebra_for_G = D_Z(fake_zebra)

        # c) Adversarial loss with the target flipped to "real" (ones): the
        #    generator is rewarded when D_Z believes the fake is a real zebra.
        loss_G_H2Z = criterion_GAN(
            pred_fake_zebra_for_G,
            torch.ones_like(pred_fake_zebra_for_G)
        )

        # d) Translate the fake zebra back with the other generator.
        reconstructed_horse = G_Z2H(fake_zebra)

        # e) Cycle consistency: the round trip must return the original
        #    horse. With unpaired data this is what ties the fake zebra to
        #    the content of its input instead of being any random zebra.
        loss_cycle_horse = criterion_cycle(reconstructed_horse, real_horse)

        # 3. Zebra -> Horse -> Zebra
        # TODO: Training the zebra generator
        # Solution: mirror image of step 2 with the roles swapped. G_Z2H
        # makes a fake horse, D_H judges it against a "real" target, and
        # G_H2Z must map it back to the original zebra.
        fake_horse = G_Z2H(real_zebra)
        pred_fake_horse_for_G = D_H(fake_horse)

        loss_G_Z2H = criterion_GAN(
            pred_fake_horse_for_G,
            torch.ones_like(pred_fake_horse_for_G)
        )

        reconstructed_zebra = G_H2Z(fake_horse)

        loss_cycle_zebra = criterion_cycle(reconstructed_zebra, real_zebra)

        loss_GAN = loss_G_H2Z + loss_G_Z2H

        loss_cycle = loss_cycle_horse + loss_cycle_zebra

        loss_G = (
            loss_GAN
            + lambda_cycle * loss_cycle
            + lambda_identity * loss_identity
        )

        loss_G.backward()
        optimizer_G.step()

        # 2. TODO: Update horse discriminator
        optimizer_D_H.zero_grad(set_to_none=True)

        # Solution: D_H should score real horses as real, so run it on the
        # real horse and regress its patch scores toward 1. The provided code
        # below adds the matching term that pushes (buffered, detached) fake
        # horses toward 0. zero_grad above also clears the gradients that the
        # generator update left on D_H.
        pred_real_horse = D_H(real_horse)

        loss_D_H_real = criterion_GAN(
            pred_real_horse,
            torch.ones_like(pred_real_horse)
        )

        buffered_fake_horse = fake_horse_buffer.push_and_pop(
            fake_horse
        )

        pred_fake_horse = D_H(buffered_fake_horse.detach())

        loss_D_H_fake = criterion_GAN(
            pred_fake_horse,
            torch.zeros_like(pred_fake_horse)
        )

        loss_D_H = 0.5 * (loss_D_H_real + loss_D_H_fake)

        loss_D_H.backward()
        optimizer_D_H.step()

        # 3. TODO: Update zebra discriminator
        optimizer_D_Z.zero_grad(set_to_none=True)

        # Solution: same as the horse discriminator, for the zebra domain:
        # real zebras are regressed toward 1, and the provided code below
        # regresses the buffered fake zebras toward 0.
        pred_real_zebra = D_Z(real_zebra)

        loss_D_Z_real = criterion_GAN(
            pred_real_zebra,
            torch.ones_like(pred_real_zebra)
        )

        buffered_fake_zebra = fake_zebra_buffer.push_and_pop(
            fake_zebra
        )

        pred_fake_zebra = D_Z(buffered_fake_zebra.detach())

        loss_D_Z_fake = criterion_GAN(
            pred_fake_zebra,
            torch.zeros_like(pred_fake_zebra)
        )

        loss_D_Z = 0.5 * (loss_D_Z_real + loss_D_Z_fake)

        loss_D_Z.backward()
        optimizer_D_Z.step()

        # 4. Accumulate metrics
        running['G'] += loss_G.item()
        running['D_H'] += loss_D_H.item()
        running['D_Z'] += loss_D_Z.item()
        running['cycle'] += loss_cycle.item()
        running['identity'] += loss_identity.item()

        progress.set_postfix({
            'G': f'{loss_G.item():.3f}',
            'D_H': f'{loss_D_H.item():.3f}',
            'D_Z': f'{loss_D_Z.item():.3f}'
        })

    # Average losses across the epoch.
    num_batches = len(dataloader)

    for key in running:
        running[key] /= num_batches
        history[key].append(running[key])

    print(
        f'\nEpoch {epoch}/{epochs} | '
        f'G: {running['G']:.4f} | '
        f'D_H: {running['D_H']:.4f} | '
        f'D_Z: {running['D_Z']:.4f} | '
        f'Cycle: {running['cycle']:.4f} | '
        f'Identity: {running['identity']:.4f}'
    )

    # Save visual progress and checkpoints.
    if epoch % save_every == 0 or epoch == epochs - 1 or epoch == 1:
        save_translations(G_H2Z, G_Z2H, real_horse[0].unsqueeze(0), real_zebra[0].unsqueeze(0), epoch)
        save_checkpoint(G_H2Z, G_Z2H, D_H, D_Z, optimizer_G, optimizer_D_H, optimizer_D_Z, epoch)

print('Training complete.')

Horses: 1067
Zebras: 1334


Epoch 1/20: 100%|██████████| 1334/1334 [00:54<00:00, 24.30it/s, G=2.218, D_H=0.152, D_Z=0.190]



Epoch 1/20 | G: 2.5265 | D_H: 0.3405 | D_Z: 0.3440 | Cycle: 0.5841 | Identity: 0.5480


Epoch 2/20: 100%|██████████| 1334/1334 [00:54<00:00, 24.59it/s, G=2.053, D_H=0.169, D_Z=0.229]



Epoch 2/20 | G: 2.2087 | D_H: 0.2713 | D_Z: 0.2639 | Cycle: 0.5257 | Identity: 0.5007


Epoch 3/20: 100%|██████████| 1334/1334 [00:56<00:00, 23.43it/s, G=2.033, D_H=0.148, D_Z=0.270]


... (31 outputs left out)

Epoch 19/20: 100%|██████████| 1334/1334 [00:58<00:00, 22.96it/s, G=2.104, D_H=0.253, D_Z=0.085]



Epoch 19/20 | G: 2.1241 | D_H: 0.2159 | D_Z: 0.1369 | Cycle: 0.4387 | Identity: 0.4010


Epoch 20/20: 100%|██████████| 1334/1334 [01:00<00:00, 21.99it/s, G=1.948, D_H=0.092, D_Z=0.080]



Epoch 20/20 | G: 2.1159 | D_H: 0.2119 | D_Z: 0.1339 | Cycle: 0.4350 | Identity: 0.4009
Training complete.
